# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** One row of the table being built is one line item. This means it will be one item within one order, plus its `order` and `vendor_id`, and the vendor's `name` and `zone` after the two are joined.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


`pd.json_normalize` turns the `orders_json`, that is nested, into a flat table. Then `record_path='lines'` tells it to look inside each order's `lines` list and make one row for every item in the list, so each row holds an `item` and `qty`. `meta=['order', 'vendor_id']` copies `order` and `vendor_id` from the parent order onto every row  produced, so the line item still knows which specific order and vendor it came from.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors_df = pd.DataFrame(vendors_json)

joined = lines.merge(vendors_df, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']
unmatched_units = unmatched.groupby('vendor_id')['qty'].sum()

print(joined['_merge'].value_counts())
print(unmatched_units if not unmatched_units.empty else "No unmatched vendors")

_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64
vendor_id
V-07    3
Name: qty, dtype: int64


The first line turns the vendor list into a table so we can merge it onto `lines`. Then, the merge attaches vendor information to each item row, matching on `vendor_id`. I used a left join, so item rows are kept even if the vendor isn't in the vendor list. `indicator=True` adds a `_merge` column that records what happened to each row. `both` indicates the vendor and item row were found, `left_only` indicates the vendor was not found. Any row that was unmatched gets a `NaN` in both name and zone.

Then, `unmatched` filters the `joined` table to only the `left_only` rows, so only the items whose vendor was not found. Then, I group those rows by vendor_id and sum `qty`to give the total units corresponding to each unmatched vendor.

Finally, the two print lines report the results from `unmatched`. The first shows how many rows matched and how many didn't. The second shows the units per unmatched vendors, or "no unmatched vendors". This showed that 3 rows were matched, and one row was unmatched. 3 units corresponding to `V-07` were unmatched.




### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
joined['name'] = joined['name'].fillna(joined['vendor_id'] + ' (unmatched)')

qty_by_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False)

qty_by_vendor

,qty
name,
Hoos Burgers,3
V-07 (unmatched),3
Cav Merch,1


Unmatched vendors have `NaN` as their `name`, and `groupby` drops `NaN` keys automatically, which would lead to losing V-07's 3 units, which would not be ideal. Because of this, I filled the missing name with its `vendor_id` to keep the unmatched vendor visible, and this allows for the totals to still add up to 7.

To do this, I used `fillna`. `joined['vendor_id'] + ' (unmatched)'` creates a label (V-07 (unmatched)) for rows that had a missing value for `name`. Matched rows kept their actual names, so only the unmatched vendor got the label that tells us which vendor it is and that it wasn't found in the vendor list.

Then, `joined.groupby('name')['qty'].sum()`adds up the quantities for each name, with `.sort_values(ascending=False)` showing the biggest totals first. This shows that both Hoos Burger and V-07 sold 3 items, and Cav Merch sold 1.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_totals = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)

print(zone_totals)
print("zone total:", zone_totals.sum(), "| overall total:", joined['qty'].sum())
print("units with no zone:", joined.loc[joined['zone'].isna(), 'qty'].sum())

zone
A    4
Name: qty, dtype: int64
zone total: 4 | overall total: 7
units with no zone: 3


The first line groups the `joined` table by `zone` and adds up `qty` within each group. Then it sorts it so the biggest zone is first. `groupby` drops rows whose key is `NaN`, which makes it so V-07's row, which has no zone after the left join, is not included. This leaves us with only zone A, which has 4 units.

The first print shows the zone table, and the second compares the two totals. `zone_totals.sum()` shows the sum of the zone table, whereas `joined['qty'].sum()` shows the sum of every row regardless of zone. This shows us that something was dropped in the zone table. The third print finds the dropped units. `joined['zone'].isna()` finds the rows with no zone. Then,  `.loc[..., 'qty']` selects the `qty` column for just those rows, and `.sum()` adds them up, giving 3. This shows that the gap between the zone total and the overall totral comes from `V-07`.





### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

safe = [{**o, 'lines': o.get('lines') or [{'item': None, 'qty': None}]} for o in ragged_json]

ragged_df = pd.json_normalize(safe, record_path='lines', meta=['order', 'vendor_id'])

ragged_df['qty'] = ragged_df['qty'].astype('Int64')

ragged_df = ragged_df[['order', 'vendor_id', 'item', 'qty']]
ragged_df

,order,vendor_id,item,qty
0,4,V-01,Soda,2
1,5,V-10,None,<NA>
2,6,V-01,Fries,<NA>


The first line builds a cleaned copy of `ragged_json`. For each order, `o.get('lines')` returns the order's list of lines if there is one, and returns `None` if the key is absent. `or` substitutes a one-item placeholder list, with `item` and `qty` set to None, if lines are missing or entirely empty. This is necessary because `record_path='lines'` will raise an error if an order has no lines key. Now, because there is a placeholder, the order will be a part of the new DataFrame and will not crash the cell or be removed. Additionally, `{**o, 'lines': ...}` copies every field of the order and only overwrites lines with the safe version. This makes it so the original `ragged_json` is not impacted.

The next line flattens the cleaned data so one row is one line, and `order` and `vendor_id ` is copied onto each row. Because order 6 was missing the `qty` key, pandas fills that cell with `NaN` automatically.

The `astype('Int64')` helps to keep a missing quantity different from a zero. The default `qty` column was float, because `NaN` forces floats. By converting it to `Int64`, pandas' nullable integer type, missing values become `<NA>`, and real numbers are integers (0 will show up as 0).

Finally, the last lines reorder the columns and display the frame.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

One row per line item is the useful shape because each row is then one thing that was sold. One row is an item, a quantity, and the order and vendor it belongs to. Everything we are wanting to count up lives at this level, which is why this shape is ideal. We can use a simple groupby and sum to find the total units per vendor or zone. If it was one row per order, the lines would be nested inside each row. We would have to search within the lists for every order to find the total quantities that are a single groupby with one row per line item. It also keeps the most detail. It's possible to add line items up to get order totals, but we can't go back from order totals to the items



**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A missing quantity means that the number of items sold is unknown. A quantity of zero indicates that no items were sold, a known figure. Conflating them, possibly with `fillna(0)`,will lead to different things breaking later on. One way is that totals and averages become wrong without any error being raised. A vendor with several unknown quantities could be undercounted because the unknown quantities would be counted as 0 rather than a missing value. Averages per line would be dragged down by the fake zeros, changing how data is interpreted. Additionally, it would be problematic if you want to find data quality problems. You wouldn't be able to filter for lines where the quantity is missing, and this gap would be invisible.